### Блок 1. Импорт файлов и библиотек

In [1]:
import pandas as pd, re
from rapidfuzz import process, fuzz

raw = pd.read_csv('data/raw_positions.csv', sep=';')
cls = pd.read_csv('data/classifier.csv', sep=';')
lbl = pd.read_csv('data/labeled_sample.csv', sep=';')

### Блок 2. Нормализация и очистка текста

In [2]:
syn = {
    r'\bпто\b': 'производственно-технического отдела', r'\bэгс\b|эл\.?газосварщик': 'электрогазосварщик',
    r'\bавтокран\w*': 'крана автомобильного', r'\bбульдозерист\w*': 'машинист бульдозера',
    r'\bшофер\w*': 'водитель', r'\bразнорабочий\w*': 'подсобный рабочий',
    r'производитель работ\b': 'прораб', r'специалист по сметам': 'сметчик', r'\bкрановщик\b': 'машинист крана',
    r'\bмаш\.|\bмаш\b': 'машинист', r'\bнач\.|\bнач\b': 'начальник', r'\bстроит\.|\bстроит\b': 'строительных',
    r'\bмех\.|\bмех\b': 'механического', r'\bруч\.|\bруч\b': 'ручного'
}

def clean(t):
    t = re.sub(r'\(.*?\)|[\"«][^\"»]+[\"»]|\b(ооо|ао|зао)\b|\d+\s*(?:разряд\w*|категори\w*|кат\.?|р-да)', '', str(t).lower().replace('ё', 'е'))
    for k, v in syn.items(): t = re.sub(k, v, t)
    return ' '.join(re.findall(r'[а-яa-z]+', t))

cls['clean'] = cls['Наименование должности по классификатору'].apply(clean)
lbl['clean'] = lbl['Исходное наименование должности'].apply(clean)

In [3]:
lbl

,id,Исходное наименование должности,Правильный код,Наименование по классификатору,clean
0,1,Кровельщик по стальным кровлям,КЛС-012,Кровельщик по стальным кровлям,кровельщик по стальным кровлям
1,2,Инженер по качеству 5 разряда,КЛС-044,Инженер по качеству,инженер по качеству
2,3,Мастер строительных и монтажных работ 1 категории,КЛС-047,Мастер строительных и монтажных работ,мастер строительных и монтажных работ
3,11,Бетонщик,КЛС-004,Бетонщик,бетонщик
4,20,Плотник 5 разряда,КЛС-006,Плотник,плотник
5,26,"Нач. участка АО ""МостСтрой""",КЛС-048,Начальник участка,начальник участка
6,27,Газоорезчик,КЛС-022,Газорезчик,газоорезчик
7,29,Машинист буровой установки,КЛС-032,Машинист буровой установки,машинист буровой установки
8,33,ПЛОТНИК,КЛС-006,Плотник,плотник
9,43,Фрезеровщик,КЛС-053,Фрезеровщик,фрезеровщик


### Блок 3. Функция для сопоставление по расстоянию Левенштейна

In [4]:
process.extractOne("абоба", cls['clean'], scorer=fuzz.token_sort_ratio)

('прораб', 36.36363636363637, 45)

(лучшая_строка, балл, индекс_в_таблице)

In [5]:
def match(text, thr=60):
    res = process.extractOne(text, cls['clean'], scorer=fuzz.token_sort_ratio)
    if not res or res[1] < thr: return 'НЕТ СООТВЕТСТВИЯ', '', round(1 - res[1]/100, 2)
    row = cls.iloc[res[2]]
    return row['Код'], row['Наименование должности по классификатору'], round(res[1]/100, 2)

### Блок 4. Оценка результатов

In [6]:
lbl[['pred_code', 'pred_name', 'conf']] = lbl['clean'].apply(lambda x: pd.Series(match(x)))
acc = (lbl['pred_code'] == lbl['Правильный код']).mean()
print(f'Accuracy на labeled_sample: {acc:.2%}')
lbl[['Исходное наименование должности', 'Правильный код', 'pred_code', 'conf']].head(10)

Accuracy на labeled_sample: 100.00%


,Исходное наименование должности,Правильный код,pred_code,conf
0,Кровельщик по стальным кровлям,КЛС-012,КЛС-012,1.00
1,Инженер по качеству 5 разряда,КЛС-044,КЛС-044,1.00
2,Мастер строительных и монтажных работ 1 категории,КЛС-047,КЛС-047,1.00
3,Бетонщик,КЛС-004,КЛС-004,1.00
4,Плотник 5 разряда,КЛС-006,КЛС-006,1.00
5,"Нач. участка АО ""МостСтрой""",КЛС-048,КЛС-048,1.00
6,Газоорезчик,КЛС-022,КЛС-022,0.95
7,Машинист буровой установки,КЛС-032,КЛС-032,1.00
8,ПЛОТНИК,КЛС-006,КЛС-006,1.00
9,Фрезеровщик,КЛС-053,КЛС-053,1.00
